In [3]:
# Imports
import pandas as pd
import xml.etree.ElementTree as ET

In [15]:
# 1.
xml_path = './data_bundestag/mdb-data.xml'
tree = ET.parse(xml_path)
root = tree.getroot()

members_data = []

for mdb in root.findall('MDB'):
    info = mdb.find('INFO')
    member = {
        'ID': info.findtext('ID'),
        'First_name': info.findtext('First_name'),
        'Last_name': info.findtext('Last_name'),
        'Acad_Title': info.findtext('Acad_Title'),
        'Date_of_birth': info.findtext('Date_of_birth'),
        'Gender': info.findtext('Gender'),
        'Party': info.findtext('Party'),
        'Marital_status': info.findtext('Marital_status'),
        'Children': info.findtext('Children'),
    }
    members_data.append(member)

# Convert to DataFrame
df_members = pd.DataFrame(members_data)

# Data type conversions
df_members['ID'] = df_members['ID'].astype(int)
df_members['Date_of_birth'] = pd.to_datetime(df_members['Date_of_birth'])
df_members['Children'] = df_members['Children'].astype(int)
df_members['Gender'] = df_members['Gender'].astype('category')
df_members['Party'] = df_members['Party'].astype('category')
df_members['Marital_status'] = df_members['Marital_status'].astype('category')
df_members['Acad_Title'] = df_members['Acad_Title'].fillna('No title')

df_members

,ID,First_name,Last_name,Acad_Title,Date_of_birth,Gender,Party,Marital_status,Children
0,11000001,Manfred,Abelein,Prof. Dr.,1930-10-20,male,CDU,no information / other,0
1,11000002,Ernst,Achenbach,Dr.,1909-04-09,male,FDP,married,3
2,11000003,Annemarie,Ackermann,None,1913-05-26,female,CDU,married,5
3,11000004,Else,Ackermann,Dr.,1933-11-06,female,CDU,no information / other,0
4,11000005,Ulrich,Adam,None,1950-06-09,male,CDU,married,2
...,...,...,...,...,...,...,...,...,...
4609,11005623,Reza,Asghari,Prof. Dr.,1961-04-01,male,CDU,married,2
4610,11005624,Andrea,Lübcke,Dr.,1978-12-09,female,GRÜNE,no information / other,2
4611,11005625,Lisa,Schubert,None,2002-09-06,diverse,DIE LINKE,no information / other,0
4612,11005626,Mayra,Vriesema,None,1999-12-25,female,GRÜNE,no information / other,0


In [19]:
# 2.
wpinfo_data = []

for mdb in root.findall('MDB'):
    member_id = mdb.find('INFO').findtext('ID')
    wplist = mdb.find('WPLIST')
    for wp in wplist.findall('WP'):
        wp_entry = {
            'ID': int(member_id),
            'WP': int(wp.findtext('WP')),
            'MDBWP_FROM': wp.findtext('MDBWP_FROM'),
            'MDBWP_UNTIL': wp.findtext('MDBWP_UNTIL'),
            'WP_BEGIN': wp.findtext('WP_BEGIN'),
        }
        wpinfo_data.append(wp_entry)

# Convert to DataFrame
df_wpinfo = pd.DataFrame(wpinfo_data)

# Data type conversions
df_wpinfo['ID'] = df_wpinfo['ID'].astype(int)
df_wpinfo['WP'] = df_wpinfo['WP'].astype(int)
df_wpinfo['MDBWP_FROM'] = pd.to_datetime(df_wpinfo['MDBWP_FROM'])
# Handle open-ended dates (marked with "/")
df_wpinfo['MDBWP_UNTIL'] = df_wpinfo['MDBWP_UNTIL'].apply(
    lambda x: pd.to_datetime(x) if x != '/' else pd.NaT
)
df_wpinfo['WP_BEGIN'] = pd.to_datetime(df_wpinfo['WP_BEGIN'])

print(f"Election Period DataFrame ({df_wpinfo.shape}):")
df_wpinfo


Election Period DataFrame ((13046, 5)):


,ID,WP,MDBWP_FROM,MDBWP_UNTIL,WP_BEGIN
0,11000001,5,1965-10-19,1969-10-19,1965-10-19
1,11000001,6,1969-10-20,1972-09-22,1969-10-20
2,11000001,7,1972-12-13,1976-12-13,1972-12-13
3,11000001,8,1976-12-14,1980-11-04,1976-12-14
4,11000001,9,1980-11-04,1983-03-29,1980-11-04
...,...,...,...,...,...
13041,11005623,21,2025-06-10,NaT,2025-03-25
13042,11005624,21,2025-07-01,NaT,2025-03-25
13043,11005625,21,2025-08-01,NaT,2025-03-25
13044,11005626,21,2025-09-01,NaT,2025-03-25


In [18]:
# 4. Summary statistics

print("=== Members Summary ===")
print(f"Total members: {len(df_members)}")
print(f"\nGender distribution:")
print(df_members['Gender'].value_counts())
print(f"\nParty distribution:")
print(df_members['Party'].value_counts())
print(f"\nAge statistics (DOB):")
print(f"Oldest member born: {df_members['Date_of_birth'].min()}")
print(f"Youngest member born: {df_members['Date_of_birth'].max()}")

print("\n=== Election Period Summary ===")
print(f"Total election period records: {len(df_wpinfo)}")
print(f"\nElection periods covered:")
print(df_wpinfo['WP'].value_counts().sort_index())


=== Members Summary ===
Total members: 4614

Gender distribution:
Gender
male       3567
female     1046
diverse       1
Name: count, dtype: int64

Party distribution:
Party
SPD          1451
CDU          1441
FDP           491
GRÜNE         344
CSU           285
AfD           199
OTHER         175
DIE LINKE     174
PDS            54
Name: count, dtype: int64

Age statistics (DOB):
Oldest member born: 1875-12-14 00:00:00
Youngest member born: 2002-09-06 00:00:00

=== Election Period Summary ===
Total election period records: 13046

Election periods covered:
WP
1     474
2     558
3     562
4     580
5     559
6     556
7     549
8     553
9     549
10    576
11    702
12    699
13    693
14    699
15    628
16    642
17    652
18    658
19    750
20    772
21    635
Name: count, dtype: int64
